In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../processed/BPS_20_26_Waldinei.csv", sep=";", encoding="utf-8")
print(df.shape)
print(df.columns.tolist())

(367003, 45)
['ano_compra', 'cnpj_instituicao', 'sg_uf', 'ds_esfera', 'dt_compra', 'dt_insercao', 'validade_compra', 'co_catmat', 'ds_item', 'co_pdm', 'co_grupo', 'no_grupo', 'co_classe', 'no_classe', 'fg_generico', 'tp_compra', 'sg_unidade_medida', 'cnpj_fornecedor', 'no_fornecedor', 'cnpj_fabricante', 'no_fabricante', 'qt_medicamento', 'ds_observacao', 'no_instituicao', 'no_municipio', 'un_medida_capacidade', 'no_pdm', 'nu_processo_compra', 'nu_ata', 'un_fornecimento', 'registro_anvisa', 'modalidade', 'vl_capacidade', 'vl_preco_unitario', 'vl_preco_total', 'co_seq_bps', 'ano_arquivo_origem', 'razao_preco_mediana', 'log10_razao', 'ordem_grandeza', 'vl_preco_unitario_corrigido', 'vl_preco_total_corrigido', 'vl_preco_unitario_bruto', 'vl_preco_total_bruto', 'flag_preco_corrigido']


## Investigação: Modalidade de Compra (Pergunta de Negócio 6)

In [ ]:
print("=== Valores únicos de 'modalidade' ===")
print(df["modalidade"].value_counts(dropna=False))

print("\n=== Cruzamento modalidade x tp_compra (amostra) ===")
print(
    df.groupby(["tp_compra", "modalidade"])
    .size()
    .reset_index(name="contagem")
    .sort_values("contagem", ascending=False)
    .head(20)
)

=== Valores únicos de 'modalidade' ===
modalidade
Pregão                          332087
Registro de Preços               18007
Dispensa de Licitação            13599
Tomada de Preços                  1864
Concorrência                       710
Inexigibilidade de Licitação       417
Concurso                           194
Convite                             70
Leilão                              54
Diálogo Competitivo                  1
Name: count, dtype: int64

=== Cruzamento modalidade x tp_compra (amostra) ===
         tp_compra                    modalidade  contagem
7   ADMINISTRATIVA                        Pregão    328806
8   ADMINISTRATIVA            Registro de Preços     16675
3   ADMINISTRATIVA         Dispensa de Licitação     12268
14        JUDICIAL                        Pregão      3281
9   ADMINISTRATIVA              Tomada de Preços      1864
15        JUDICIAL            Registro de Preços      1332
11        JUDICIAL         Dispensa de Licitação      1331
0   ADMIN

In [ ]:
analise_modalidade = (
    df.groupby("modalidade")
    .agg(
        registros=("vl_preco_total", "count"),
        valor_total=("vl_preco_total", "sum"),
        quantidade_total=("qt_medicamento", "sum"),
    )
    .reset_index()
)

analise_modalidade["preco_medio_ponderado"] = (
    analise_modalidade["valor_total"] / analise_modalidade["quantidade_total"]
)
analise_modalidade["pct_valor_total"] = (
    analise_modalidade["valor_total"] / analise_modalidade["valor_total"].sum() * 100
)

analise_modalidade = analise_modalidade.sort_values("valor_total", ascending=False)
print(analise_modalidade.to_string())

                     modalidade  registros   valor_total  quantidade_total  preco_medio_ponderado  pct_valor_total
7                        Pregão     332087  4.384304e+10       58045686099               0.755319        88.920378
8            Registro de Preços      18007  3.742638e+09        4980677571               0.751431         7.590641
3         Dispensa de Licitação      13599  1.634571e+09        1731287939               0.944136         3.315161
5  Inexigibilidade de Licitação        417  6.614054e+07          13667584               4.839227         0.134143
9              Tomada de Preços       1864  1.059172e+07          14311462               0.740086         0.021482
0                  Concorrência        710  3.566332e+06           7037571               0.506756         0.007233
1                      Concurso        194  2.637446e+06           5277970               0.499708         0.005349
6                        Leilão         54  2.002841e+06           9191181      

## Validação: CATMAT vs. Descrição (Pergunta de Negócio 3)

In [ ]:
top10_catmat = (
    df.groupby("co_catmat")
    .agg(
        valor_total=("vl_preco_total", "sum"),
        descricoes_distintas=("ds_item", "nunique"),
        descricao_exemplo=("ds_item", "first"),
    )
    .reset_index()
    .sort_values("valor_total", ascending=False)
    .head(10)
)

top10_item = (
    df.groupby("ds_item")["vl_preco_total"]
    .sum()
    .reset_index()
    .sort_values("vl_preco_total", ascending=False)
    .head(10)
)

print("=== TOP 10 POR co_catmat ===")
print(top10_catmat.to_string())
print("\n=== TOP 10 POR ds_item ===")
print(top10_item.to_string())
print(
    f"\nCATMATs com mais de 1 descrição distinta: {(df.groupby('co_catmat')['ds_item'].nunique() > 1).sum()}"
)

=== TOP 10 POR co_catmat ===
      co_catmat   valor_total  descricoes_distintas                                                                                                                                                                    descricao_exemplo
8216     439252  1.271712e+09                     1                                                                                                            NUSINERSENA, CONCENTRAÇÃO:2,4 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL
7067     432908  1.084856e+09                     1                                                                                                                                                   DAPAGLIFLOZINA, CONCENTRAÇÃO:10 MG
9882     452740  9.224414e+08                     1                                                                                                                       OMALIZUMABE, CONCENTRAÇÃO:150 MG, FORMA FARMACÊUTICA:INJETÁVEL
4451     400563  8.819573e+08          

## Análise: Top 10 Fabricantes (Pergunta de Negócio 4)

In [ ]:
top10_fabricantes = (
    df.groupby("no_fabricante")
    .agg(valor_total=("vl_preco_total", "sum"), registros=("vl_preco_total", "count"))
    .reset_index()
    .sort_values("valor_total", ascending=False)
    .head(10)
)

top10_fabricantes["pct_valor_total"] = (
    top10_fabricantes["valor_total"] / df["vl_preco_total"].sum() * 100
)
print(top10_fabricantes.to_string())
print(
    f"\nConcentração dos Top 10 fabricantes: {top10_fabricantes['pct_valor_total'].sum():.2f}% do valor total"
)

                                                    no_fabricante   valor_total  registros  pct_valor_total
1534                                      NOVARTIS BIOCIENCIAS SA  3.919206e+09       4266         7.948749
1072                              JANSSEN-CILAG FARMACEUTICA LTDA  2.079847e+09        865         4.218247
170                                   ASTRAZENECA DO BRASIL LTDA.  1.857312e+09       1622         3.766912
1866                              SANOFI MEDLEY FARMACEUTICA LTDA  1.585869e+09       4013         3.216384
36                             ACHE LABORATORIOS FARMACEUTICOS SA  1.576320e+09       4281         3.197018
507                CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA  1.538333e+09      22397         3.119975
676                                                       EMS S/A  1.519386e+09      18438         3.081548
319   BOEHRINGER INGELHEIM DO BRASIL QUIMICA E FARMACEUTICA LTDA.  1.486342e+09       2050         3.014529
1694                        